In [1]:
"""
Minimal on-disk ANN index wrapper around USearch.

    pip install usearch numpy

Usage:
    store = AnnStore("my_index.usearch", dim=128)   # loads if the file exists
    store.insert(42, my_int8_array)
    store.save()
    keys, dists = store.search(query_array, k=5)
"""
import json
import os

import numpy as np
from usearch.index import Index


class AnnStore:
    def __init__(self, path, dim, metric="l2sq", dtype="i8",
                 connectivity=16, expansion_add=128, expansion_search=64):
        # dtype: "i8" (int8), "f16", "f32", or "b1" (bit-packed, use metric="hamming")
        self.path = path
        self.meta_path = path + ".meta.json"
        self.dim = dim
        self.dtype = dtype
        self.np_dtype = {"i8": np.int8, "f16": np.float16, "f32": np.float32}.get(dtype)

        self.index = Index(
            ndim=dim,
            metric=metric,
            dtype=dtype,
            connectivity=connectivity,
            expansion_add=expansion_add,
            expansion_search=expansion_search,
        )

        if os.path.exists(path):
            self._check_manifest()
            self.index.load(path)  # use self.index.view(path) to memory-map instead

    # ---------- persistence ----------
    def save(self):
        tmp = self.path + ".tmp"
        self.index.save(tmp)
        os.replace(tmp, self.path)  # atomic: a crash never leaves a half-written file
        with open(self.meta_path, "w") as f:
            json.dump({"dim": self.dim, "dtype": self.dtype, "count": len(self.index)}, f)

    def _check_manifest(self):
        if os.path.exists(self.meta_path):
            with open(self.meta_path) as f:
                meta = json.load(f)
            if meta["dim"] != self.dim or meta["dtype"] != self.dtype:
                raise ValueError(
                    f"Index on disk has dim={meta['dim']}, dtype={meta['dtype']}; "
                    f"requested dim={self.dim}, dtype={self.dtype}. Rebuild the index."
                )

    # ---------- operations ----------
    def _prep(self, arr):
        arr = np.asarray(arr)
        if self.np_dtype is not None:
            arr = arr.astype(self.np_dtype, copy=False)
        if arr.shape[-1] != self.dim:
            raise ValueError(f"Expected dimension {self.dim}, got {arr.shape[-1]}")
        return arr

    def insert(self, key, array):
        """Insert one vector under an integer key."""
        self.index.add(int(key), self._prep(array))

    def insert_many(self, keys, arrays):
        """Insert a batch (2D array, one row per key). Much faster than looping."""
        self.index.add(np.asarray(keys, dtype=np.uint64), self._prep(arrays))

    def search(self, array, k=10):
        """Return (keys, distances) of the k nearest neighbours of one vector."""
        matches = self.index.search(self._prep(array), k)
        return matches.keys, matches.distances

    def remove(self, key):
        """Mark a key as deleted (space is only reclaimed by rebuilding)."""
        self.index.remove(int(key))

    def __len__(self):
        return len(self.index)


if __name__ == "__main__" and "get_ipython" not in globals():   # demo: standalone script only, not in a notebook
    rng = np.random.default_rng(0)
    data = rng.integers(-128, 127, size=(1000, 64), dtype=np.int8)

    store = AnnStore("demo.usearch", dim=64)
    if len(store) == 0:
        store.insert_many(np.arange(1000), data)
        store.save()
        print("Built and saved index with", len(store), "vectors")
    else:
        print("Loaded index with", len(store), "vectors")

    keys, dists = store.search(data[123], k=3)
    print("Nearest keys:", keys, "distances:", dists)  # key 123 should come first

In [2]:
pip install usearch numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests

LAT_MIN, LAT_MAX = 61.40, 61.55
LON_MIN, LON_MAX = 23.60, 23.95


def normalize(lat: float, lon: float, size: int = 1000) -> tuple[int, int]:
    """Return (x, y) as ints: x = west->east, y = south->north, both in 0..size."""
    x = (lon - LON_MIN) / (LON_MAX - LON_MIN) * size
    y = (lat - LAT_MIN) / (LAT_MAX - LAT_MIN) * size
    x = max(0.0, min(float(size), x))
    y = max(0.0, min(float(size), y))
    return round(x), round(y)


def place_to_xy(query: str, size: int = 1000) -> tuple[int, int] | None:
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": query, "format": "json", "limit": 1},
        headers={"User-Agent": "hackathon/0.1 (ivars.nya@gmail.com)"},
        timeout=10,
    )
    resp.raise_for_status()
    results: list[dict[str, str]] = resp.json()
    if not results:
        return None
    return normalize(float(results[0]["lat"]), float(results[0]["lon"]), size)

if __name__ == "__main__" and "get_ipython" not in globals():   # demo: standalone script only, not in a notebook
    print(place_to_xy("Tampere Central Square"))

In [3]:
"""
Category table: category (str) -> options (str[]) and dimensions (int[]).

options[i] corresponds to dimensions[i]. Each option owns one vector dimension.
The global nextDim is the lowest dimension not yet used.
"""
import json
import os

CATCHALL = "factors within this category that aren't covered by any of the options"

# category -> {"options": [str], "dimensions": [int]}
RESERVED_DIMS = 3          # dims 0-2 hold score, x, y
table = {}
nextDim = RESERVED_DIMS


def _allocate(count):
    """Hand out `count` consecutive dimensions starting at nextDim."""
    global nextDim
    dims = list(range(nextDim, nextDim + count))
    nextDim += count
    return dims


def add_category(category, options):
    """Create a new category. The catch-all option is always placed first."""
    if category in table:
        raise ValueError(f"Category '{category}' already exists")
    options = [o for o in options if o != CATCHALL]  # avoid duplicating the catch-all
    if len(set(options)) != len(options):
        raise ValueError("Duplicate options given")
    all_options = [CATCHALL] + options
    table[category] = {"options": all_options, "dimensions": _allocate(len(all_options))}


def add_options(category, options):
    """Append options to an existing category, using dimensions from nextDim upward."""
    if category not in table:
        raise KeyError(f"Category '{category}' does not exist")
    entry = table[category]
    new = [o for o in options if o not in entry["options"]]  # skip ones already present
    if len(set(new)) != len(new):
        raise ValueError("Duplicate options given")
    entry["options"].extend(new)
    entry["dimensions"].extend(_allocate(len(new)))


def save(path="categories.json"):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(table, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)  # atomic


def load(path="categories.json"):
    """Load the table and recompute nextDim as (highest used dimension) + 1."""
    global table, nextDim
    if not os.path.exists(path):
        table, nextDim = {}, RESERVED_DIMS
        return
    with open(path, encoding="utf-8") as f:
        table = json.load(f)
    used = [d for entry in table.values() for d in entry["dimensions"]]
    nextDim = max(max(used) + 1 if used else 0, RESERVED_DIMS)


if __name__ == "__main__" and "get_ipython" not in globals():   # demo: standalone script only, not in a notebook
    load("demo_categories.json")
    if not table:
        add_category("colour", ["red", "green"])
        add_category("size", ["small", "large"])
        add_options("colour", ["blue"])
        save("demo_categories.json")
    for cat, entry in table.items():
        print(cat, list(zip(entry["dimensions"], entry["options"])))
    print("nextDim =", nextDim)

In [4]:
"""Load the tower table and look up (x, y) by code, name, or cell ID.

    from tower_lookup import TowerIndex
    towers = TowerIndex("towers.csv.gz")
    towers.get("FI-KAL-M00001")      # -> (412, 733) or None
    towers["Kalamari Centre Mast 1"] # -> (x, y), raises KeyError if missing
    towers.get(100000)               # numeric cell ID works too

Lookups are case-insensitive and ignore surrounding whitespace.
"""
import csv
import gzip


def _key(value):
    return str(value).strip().casefold()


class TowerIndex:
    def __init__(self, path=None):
        self._by_key = {}
        self._rows = 0
        if path:
            self.load(path)

    def load(self, path):
        opener = gzip.open if path.endswith(".gz") else open
        with opener(path, "rt", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                xy = (int(row["x"]), int(row["y"]))
                # one dict, three kinds of key: code, name, cell_id
                self._by_key[_key(row["code"])] = xy
                self._by_key[_key(row["name"])] = xy
                self._by_key[_key(row["cell_id"])] = xy
                self._rows += 1
        return self

    def get(self, query, default=None):
        return self._by_key.get(_key(query), default)

    def __getitem__(self, query):
        return self._by_key[_key(query)]

    def __contains__(self, query):
        return _key(query) in self._by_key

    def __len__(self):
        """Number of sites (rows loaded)."""
        return self._rows


def _in_ipython():
    try:
        get_ipython  # defined only inside IPython/Jupyter
        return True
    except NameError:
        return False


if __name__ == "__main__" and not _in_ipython():
    import sys

    if len(sys.argv) < 3:
        sys.exit("usage: python tower_lookup.py TABLE.csv[.gz] QUERY [QUERY ...]")
    idx = TowerIndex(sys.argv[1])
    print(f"Loaded {len(idx)} sites")
    for q in sys.argv[2:]:
        print(q, "->", idx.get(q))

In [5]:
towers = TowerIndex("towers.csv.gz")
print(len(towers))                    # 200000
print(towers.get("FI-PRK-M00001"))    # (96, 362)

200000
(96, 362)


In [6]:
GEMINI_API_KEY=open("gemkey.txt", encoding="utf-8").read().strip()

In [13]:
pip install google-genai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 2.5 MB/s  0:00:00

   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---------------------------------------- 0/9 [websockets]
   ---- ----------------------------------- 1/9 [truststore]
   -------- ------------------------------- 2/9 [tenacity]
  Attempting uninstall: idna
   -------- ------------------------------- 2/9 [tenacity]
    Found existing installation: idna 3.10
   -------- ------------------------------- 2/9 [tenacity]
    Uninstalling idna-3.10:
   -----


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
"""Thin wrapper around the Gemini API (google-genai SDK).

Set GEMINI_API_KEY in the environment (or server/.env) before calling.

    from gemini import ask_gemini
    print(ask_gemini("Say hi"))
"""

import os
from functools import lru_cache
from pathlib import Path

from google import genai
from google.genai import errors, types

#load_dotenv(Path(__file__).resolve().parent / ".env")

DEFAULT_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
# Tried in order when the main model is overloaded, times out or is unavailable.
FALLBACK_MODELS = [
    m.strip()
    for m in os.getenv(
        "GEMINI_FALLBACK_MODELS", "gemini-3.1-flash-lite"
    ).split(",")
    if m.strip()
]
TIMEOUT_SECONDS = float(os.getenv("GEMINI_TIMEOUT", "30"))


@lru_cache(maxsize=1)
def _client() -> genai.Client:
    api_key = GEMINI_API_KEY
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY is not set")
    return genai.Client(
        api_key=api_key,
        http_options=types.HttpOptions(timeout=int(TIMEOUT_SECONDS * 1000)),
    )


def ask_gemini(
    prompt: str,
    *,
    system: str | None = None,
    model: str = DEFAULT_MODEL,
    temperature: float | None = None,
    json_output: bool = False,
) -> str:
    """Send a prompt to Gemini and return the response text.

    system:      optional system instruction.
    json_output: ask the model to reply with JSON (parse the result with json.loads).
    """
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        response_mime_type="application/json" if json_output else None,
    )
    last_error: Exception | None = None
    for candidate in [model, *(m for m in FALLBACK_MODELS if m != model)]:
        try:
            response = _client().models.generate_content(
                model=candidate, contents=prompt, config=config
            )
            return response.text or ""
        except errors.ClientError as e:
            # Bad key / bad request won't be fixed by another model; 404 and 429 might.
            if e.code not in (404, 429):
                raise
            last_error = e
        except Exception as e:  # 5xx overload, timeouts, network errors
            last_error = e
        print(f"[gemini] {candidate} failed ({type(last_error).__name__}), trying next model")
    raise RuntimeError("All Gemini models failed") from last_error


if __name__ == "__main__" and "get_ipython" not in globals():   # demo: standalone script only, not in a notebook
    print(ask_gemini("Reply with one short sentence confirming you work."))

In [ ]:
"""
Situations table: integer key, timestamp, vector (int8 array).

Stored in SQLite, so persistence is automatic: there is no separate save/load step.
The integer key is also the key you give to the ANN index, so the two stay linked.
"""
import sqlite3
from datetime import datetime, timezone

import numpy as np


class Situations:
    def __init__(self, path="situations.db", dim=64, dtype=np.int8):
        self.dim = dim
        self.dtype = np.dtype(dtype)
        self.db = sqlite3.connect(path, check_same_thread=False)
        self.db.execute(
            """CREATE TABLE IF NOT EXISTS situations (
                   id        INTEGER PRIMARY KEY AUTOINCREMENT,
                   timestamp TEXT NOT NULL,   -- ISO 8601, UTC
                   vector    BLOB             -- NULL until vectorized; dim * itemsize raw bytes
               )"""
        )
        self.db.commit()

    # ---------- helpers ----------
    def _to_blob(self, vector):
        arr = np.asarray(vector)
        if arr.shape != (self.dim,):
            raise ValueError(f"Expected shape ({self.dim},), got {arr.shape}")
        info = np.iinfo(self.dtype)
        if arr.min() < info.min or arr.max() > info.max:
            raise ValueError(f"Values must fit in {self.dtype} ({info.min}..{info.max})")
        return arr.astype(self.dtype).tobytes()

    def _from_blob(self, blob):
        if blob is None:
            return None
        return np.frombuffer(blob, dtype=self.dtype).copy()

    # ---------- operations ----------
    def add(self, vector=None, timestamp=None):
        """Insert a situation (vector optional) and return its new integer key."""
        if timestamp is None:
            timestamp = datetime.now(timezone.utc)
        ts = timestamp.isoformat() if isinstance(timestamp, datetime) else str(timestamp)
        cur = self.db.execute(
            "INSERT INTO situations (timestamp, vector) VALUES (?, ?)",
            (ts, None if vector is None else self._to_blob(vector)),
        )
        self.db.commit()
        return cur.lastrowid

    def pending(self):
        """Keys of situations that have no vector yet."""
        rows = self.db.execute(
            "SELECT id FROM situations WHERE vector IS NULL ORDER BY id"
        ).fetchall()                  # materialized: we write while looping
        return [r[0] for r in rows]

    def set_vector(self, key, vector):
        self.db.execute(
            "UPDATE situations SET vector = ? WHERE id = ?",
            (self._to_blob(vector), int(key)),
        )
        self.db.commit()

    def get_timestamp(self, key):
        """int(key) because the ANN index returns numpy uint64, which sqlite can't bind."""
        row = self.db.execute(
            "SELECT timestamp FROM situations WHERE id = ?", (int(key),)
        ).fetchone()
        return None if row is None else row[0]

    def get(self, key):
        """Return (key, timestamp, vector) or None if the key doesn't exist."""
        row = self.db.execute(
            "SELECT id, timestamp, vector FROM situations WHERE id = ?", (key,)
        ).fetchone()
        return None if row is None else (row[0], row[1], self._from_blob(row[2]))

    def all(self):
        """Yield (key, timestamp, vector) for every row, e.g. to rebuild the ANN index."""
        for key, ts, blob in self.db.execute(
            "SELECT id, timestamp, vector FROM situations ORDER BY id"
        ):
            yield key, ts, self._from_blob(blob)

    def delete(self, key):
        self.db.execute("DELETE FROM situations WHERE id = ?", (key,))
        self.db.commit()

    def __len__(self):
        return self.db.execute("SELECT COUNT(*) FROM situations").fetchone()[0]

    def close(self):
        self.db.close()


if __name__ == "__main__" and "get_ipython" not in globals():   # demo: standalone script only, not in a notebook
    db_path = Path.cwd() / "processing" / "demo2_situations.db"
    s = Situations(str(db_path), dim=8)
    k = s.add(np.array([0, 1, 0, 0, -3, 0, 2, 0]))
    print("Inserted key", k, "| total rows:", len(s))
    print(s.get(k))
    s.close()

In [9]:
def get_categories():
    """Return every category name, in the order they were added."""
    return list(table)
 
 
def get_options(category, with_dimensions=False):
    """
    Return every option of a category (the catch-all first), as a new list.
    With with_dimensions=True, return (option, dimension) pairs instead.
    """
    if category not in table:
        raise KeyError(f"Category '{category}' does not exist")
    entry = table[category]
    if with_dimensions:
        return list(zip(entry["options"], entry["dimensions"]))
    return list(entry["options"])

In [ ]:
import os
from pathlib import Path

# Run the notebook from the project folder. separation.py writes to the same places:
#   <project>/situation_texts/<key>.txt  and  <project>/processing/demo2_situations.db
PROJECT_ROOT = Path.cwd()
TEXT_DIR = str(PROJECT_ROOT / "situation_texts")

def text_path(key):
    return os.path.join(TEXT_DIR, f"{int(key)}.txt")

def save_text(key, text):
    os.makedirs(TEXT_DIR, exist_ok=True)
    path = text_path(key)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(text)
    os.replace(tmp, path)          # atomic: no half-written text files

def load_text(key):
    with open(text_path(key), encoding="utf-8", newline="") as f:
        return f.read()            # raises FileNotFoundError if missing

def delete_text(key):
    try:
        os.remove(text_path(key))
    except FileNotFoundError:
        pass

def add_situation(situations, text):
    key = situations.add()         # SQLite creates the key first...
    save_text(key, text)           # ...then the text goes in <key>.txt
    return key

In [11]:
from pathlib import Path

print("cwd:", Path.cwd())

# Look for a folder named "processing" in cwd and each parent above it
candidates = [p / "processing" for p in [Path.cwd(), *Path.cwd().parents]]
for c in candidates:
    print(c, "->", "EXISTS" if c.exists() else "missing")

cwd: c:\Users\ivars\OneDrive\Skrivbord\Hackathon
c:\Users\ivars\OneDrive\Skrivbord\Hackathon\processing -> EXISTS
c:\Users\ivars\OneDrive\Skrivbord\processing -> missing
c:\Users\ivars\OneDrive\processing -> missing
c:\Users\ivars\processing -> missing
c:\Users\processing -> missing
c:\processing -> missing


In [29]:
import sys, platform
print(platform.system(), sys.executable)

Windows c:\Users\ivars\AppData\Local\Programs\Python\Python314\python.exe


In [ ]:
# Create the objects the driver and find_similar use. situations.dim must equal store.dim.
# db_path and TEXT_DIR must match DB_PATH and TEXT_DIR in separation.py.
DIM = 128
db_path = PROJECT_ROOT / "processing" / "demo2_situations.db"
situations = Situations(db_path, dim=DIM, dtype=np.int16)
store = AnnStore("situations.usearch", dim=DIM, dtype="f32")   # "i8" would wrap values > 127
load()          # categories.json -> table, nextDim (towers was created above)
print(f"{len(situations)} situation(s) in {db_path}, {len(store)} in the index")


In [12]:
import json
import re
import traceback
import numpy as np

RESERVED_DIMS = 3             # dims 0-2 hold score, x, y; categories start after them

COORD_RE = re.compile(
    r"^\s*[\(\[]?\s*([-+]?\d*\.?\d+)\s*(?:,|\s)\s*([-+]?\d*\.?\d+)\s*[\)\]]?\s*$"
)

SCORE_THRESHOLD = 500         # first-stub score above this -> try adding a category
MAX_EXPANSIONS = 2            # category-expansion retries per situation
OPTION_THRESHOLD = 500        # catch-all score above this -> try adding options
MAX_OPTION_EXPANSIONS = 1     # option-expansion retries per category per situation

NEW_OPTIONS_PROMPT = (
    "Within the category of {category} (current options: {options}), there may be "
    "significant aspects of the situation in the text below that are not covered by "
    "the options. Create one or a small number of extra options to represent them. "
    "Keep them short. As an example, for category weather, add thunder, heat, or cold. "
    "Only add what is relevant for this particular situation; if nothing is missing, "
    "return an empty array. Return only a JSON array of strings.\n\n"
    "--- situation ---\n{text}\n--- end situation ---"
)

NEW_CATEGORY_PROMPT = (
    "The situation in the text below is not adequately covered by the existing "
    "categories: {categories}. Create a new category and a small number of options "
    "to represent it. As an example, for category weather, add thunder, heat, or cold. "
    "Don't add options preemptively, only add what is relevant for this particular "
    "situation. Return only a JSON array of strings: the category name first, then "
    "its options.\n\n"
    "--- situation ---\n{text}\n--- end situation ---"
)


def _strip_fences(raw):
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip())


def _ask_json_array(prompt):
    """For schema-expansion calls: a malformed reply means 'nothing to add'."""
    try:
        data = json.loads(_strip_fences(ask_gemini(prompt)))
    except json.JSONDecodeError:
        return []
    if not isinstance(data, list):
        return []
    return [str(s).strip() for s in data if str(s).strip()]


# ---------------------------------------------------------------- stubs

NO_LOCATION = {"", "null", "none", "unknown", "n/a"}
_xy_cache = {}                # location string -> (x, y) or None; saves repeat geocoding


def resolve_xy(location):
    """Location string -> (x, y) ints, or None if it can't be placed."""
    if location in _xy_cache:
        return _xy_cache[location]
    m = COORD_RE.match(location)
    if m:
        xy = normalize(float(m.group(1)), float(m.group(2)))   # assumes "lat, lon"
    else:
        xy = towers.get(location)
        if xy is None:
            xy = place_to_xy(location)
    _xy_cache[location] = xy
    return xy


def first_vectorization(text, categories):
    """categories: list of category names. Returns (score, x, y)."""
    prompt = (
        "Read attachment 1 and return a location if discernible. Prioritize "
        "coordinates or a specific hardware location such as a cell tower; "
        "otherwise name the general location (city, or part thereof) with as "
        "much precision as you are confident in. If no location is discernible, "
        "use null. Also return a score from 0 to "
        "1000 for how much information about the situation can't be represented "
        "in the categories in attachment 2.\n"
        'Respond with JSON only, in the form ["<location or null>", <score>].\n\n'
        "--- attachment 1 ---\n"
        f"{text}\n"
        "--- end attachment 1 ---\n\n"
        "--- attachment 2 ---\n"
        f"{', '.join(categories)}\n"
        "--- end attachment 2 ---"
    )

    data = json.loads(_strip_fences(ask_gemini(prompt)))
    location, score = data[0], int(data[1])

    # No (or unplaceable) location: x/y stay at 0 so the situation is still vectorized.
    xy = None
    if location is not None and str(location).strip().lower() not in NO_LOCATION:
        xy = resolve_xy(str(location))
    if xy is None:
        xy = (0, 0)
    return score, int(xy[0]), int(xy[1])


def general_vectorization(text, category, options):
    prompt = (
        "Within the category of attachment 1 score each of the options from 0 to 1000 on how important they are to the situation in attachment 2. It is possible that none of them are important, if so, all scores should be low. Return only comma separated integers."
        "\n--- attachment 1 ---\n"
        f"{category}\n"
        f"{', '.join(options)}\n"
        "--- end attachment 1 ---\n\n"
        "--- attachment 2 ---\n"
        f"{text}\n"
        "--- end attachment 2 ---"
    )
    nums = [int(n) for n in re.findall(r"-?\d+", ask_gemini(prompt))]
    if len(nums) != len(options):
        raise ValueError(f"Expected {len(options)} scores for '{category}', got {len(nums)}")
    return nums


def _apply(vector, contribution, dims=None):
    """Add a stub's result (dict or array) into the working vector, in place.

    If dims is given, contribution[i] is added to vector[dims[i]].
    """
    if isinstance(contribution, dict):
        for dim, value in contribution.items():
            vector[dim] += value
    elif dims is not None:
        dims = np.asarray(dims, dtype=np.intp)
        values = np.asarray(contribution, dtype=np.int32)
        np.add.at(vector, dims, values)
    else:
        vector += np.asarray(contribution, dtype=np.int32)


# ------------------------------------------------- schema expansion
# Uses table, nextDim, CATCHALL, _allocate and add_category from the category cell.
# The vector space is fixed-size; nextDim is the first unused dimension.

def _room(dim):
    """Free dimensions left."""
    return dim - nextDim


def add_options_capped(category, options, dim):
    """Append options to an existing category, as many as fit. Returns those added.
    (Named differently so it doesn't overwrite the category cell's add_options.)"""
    entry = table[category]
    new = [o for o in dict.fromkeys(options)
           if o != CATCHALL and o not in entry["options"]]
    new = new[:max(_room(dim), 0)]
    if new:
        entry["dimensions"].extend(_allocate(len(new)))
        entry["options"].extend(new)
    return new


def add_new_category(name, options, dim):
    """Create a category if the catch-all plus at least one option fit."""
    options = [o for o in dict.fromkeys(options) if o != CATCHALL]
    options = options[:max(_room(dim) - 1, 0)]       # 1 slot reserved for the catch-all
    if not options:
        return False
    add_category(name, options)
    return True


def expand_categories(text, categories, dim):
    """Called only from the first-vectorization retry loop.
    Adds at most one new category. Returns True if one was added."""
    if _room(dim) <= 1:
        print("Vector space full; cannot add category")
        return False

    proposal = _ask_json_array(NEW_CATEGORY_PROMPT.format(
        categories=", ".join(categories), text=text))
    if len(proposal) < 2:
        return False

    name, options = proposal[0], proposal[1:]
    if name in table:                        # reused name: don't touch its options here
        return False
    return add_new_category(name, options, dim)


def expand_options(text, category, options, dim):
    """Called only from the general-vectorization step of `category`.
    Returns True if options were added to that category."""
    if _room(dim) <= 0:
        return False
    extra = _ask_json_array(NEW_OPTIONS_PROMPT.format(
        category=category, options=", ".join(options), text=text))
    return bool(add_options_capped(category, extra, dim))


def needs_options(options, scores):
    """Trigger: the catch-all option scored high for this category."""
    return scores[options.index(CATCHALL)] > OPTION_THRESHOLD


def score_category(text, category, dim):
    """general_vectorization for one category, expanding its options if needed.
    Returns (scores, dims) for the category's final option set."""
    options = get_options(category)
    scores = general_vectorization(text, category, options)

    for _ in range(MAX_OPTION_EXPANSIONS):
        if not needs_options(options, scores):
            break
        if not expand_options(text, category, options, dim):
            break
        options = get_options(category)                  # now longer
        scores = general_vectorization(text, category, options)

    return scores, list(table[category]["dimensions"])


# ----------------------------------------------------------- driver

def _vectorize_one(text, dim, dtype):
    # 1) first stub: text + all category names; only this step may add categories
    for attempt in range(MAX_EXPANSIONS + 1):
        categories = get_categories()
        score, x, y = first_vectorization(text, categories)

        if score <= SCORE_THRESHOLD or attempt == MAX_EXPANSIONS:
            break
        if not expand_categories(text, categories, dim):
            break                                 # nothing new, or no room left

    vector = np.zeros(dim, dtype=np.int32)
    _apply(vector[:RESERVED_DIMS], (score, x, y))

    # 2) second stub: once per category (including any just added);
    #    only this step may add options, and only to its own category
    for category in get_categories():
        scores, dims = score_category(text, category, dim)
        _apply(vector, scores, dims=dims)

    info = np.iinfo(dtype)
    return np.clip(vector, info.min, info.max).astype(dtype)


def vectorize_situations(situations, store):
    global nextDim
    if situations.dim != store.dim:
        raise ValueError(f"situations.dim={situations.dim} but store.dim={store.dim}")
    used = [d for entry in table.values() for d in entry["dimensions"]]
    if used and min(used) < RESERVED_DIMS:
        raise RuntimeError("Category dimensions overlap the reserved dims 0-2")
    nextDim = max(nextDim, RESERVED_DIMS)

    done = 0
    try:
        for key in situations.pending():          # situations whose vector is NULL
            try:
                text = load_text(key)
            except FileNotFoundError:
                print(f"Skipping situation {key}: no text file")
                continue
            try:
                vector = _vectorize_one(text, situations.dim, situations.dtype)
            except Exception:
                print(f"Situation {key} failed, left pending:")
                traceback.print_exc()
                continue

            situations.set_vector(key, vector)    # set the vector in the table
            store.insert(key, vector)             # and add it to the index
            done += 1
    finally:
        save()                                    # category table: keep any new dims
        if done:
            store.save()
    return done


In [13]:
import time
from datetime import datetime

TOP_K = 10
CANDIDATE_FACTOR = 3            # fetch k * this from the index, then re-rank by combined score
DISTANCE_SCALE_PER_DIM = 10000.0   # l2sq distance per allocated dimension at which relevance is 0.5
                                   # (~100 apart on each of a few active dims counts as a decent match)
RECENCY_HALF_LIFE_DAYS = 30.0   # age at which recency drops to 0.5
RELEVANCE_WEIGHT = 0.7
RECENCY_WEIGHT = 0.3


def vectorize_query(text):
    """Same vector as the ingest driver builds, but with a fixed schema (no expansion)."""
    categories = get_categories()

    score, x, y = first_vectorization(text, categories)
    vector = np.zeros(situations.dim, dtype=np.int32)
    _apply(vector[:3], (score, x, y))

    for category in categories:
        scores = general_vectorization(text, category, get_options(category))
        _apply(vector, scores, dims=table[category]["dimensions"])

    info = np.iinfo(situations.dtype)
    return np.clip(vector, info.min, info.max).astype(situations.dtype)


def _to_epoch(ts):
    """Accept epoch seconds, datetime, or ISO string."""
    if isinstance(ts, datetime):
        return ts.timestamp()
    if isinstance(ts, str):
        return datetime.fromisoformat(ts).timestamp()
    return float(ts)


def distance_scale():
    """l2sq distances grow as more dimensions become active, so the 0.5 point grows
    with the number of allocated dimensions (nextDim)."""
    return DISTANCE_SCALE_PER_DIM * nextDim


def relevance_score(distance):
    """Distance (0 = identical) -> (0, 1]; 0.5 at distance_scale()."""
    return 1.0 / (1.0 + float(distance) / distance_scale())


def recency_score(timestamp, now):
    """Exponential decay by age; 0.5 at RECENCY_HALF_LIFE_DAYS. Unknown time -> 0."""
    if timestamp is None:
        return 0.0
    age_days = max(now - _to_epoch(timestamp), 0.0) / 86400.0
    return 0.5 ** (age_days / RECENCY_HALF_LIFE_DAYS)


def find_similar(text, k=TOP_K, now=None):
    """find_similar("big string of text") -> best-first list of dicts:
    {key, distance, relevance, timestamp, recency, score}

    score = RELEVANCE_WEIGHT * relevance + RECENCY_WEIGHT * recency.
    Uses the notebook's `situations`, `store`, and `ct`. Read-only: nothing is
    inserted, and the schema/vector space is never expanded.
    """
    now = time.time() if now is None else now
    vector = vectorize_query(text)

    # Over-fetch by distance so recency can promote slightly farther but newer hits.
    keys, distances = store.search(vector, k * CANDIDATE_FACTOR)

    results = []
    for key, dist in zip(keys, distances):
        ts = situations.get_timestamp(key)
        rel = relevance_score(dist)
        rec = recency_score(ts, now)
        results.append({
            "key": key,
            "distance": float(dist),
            "relevance": rel,
            "timestamp": ts,
            "recency": rec,
            "score": RELEVANCE_WEIGHT * rel + RECENCY_WEIGHT * rec,
        })

    results.sort(key=lambda r: r["score"], reverse=True)
    return results[:k]

In [ ]:
# Vectorize everything still pending, e.g. the situations separation.py just added.
# Only rows whose vector is NULL are processed, so re-running this is free when nothing is new.
# (To add one by hand: add_situation(situations, "text of a situation ..."))
pending = situations.pending()
missing = [k for k in pending if not os.path.exists(text_path(k))]
print(f"{len(pending)} pending situation(s); {len(missing)} without a text file in {TEXT_DIR}")
if pending:
    done = vectorize_situations(situations, store)
    print(f"Vectorized {done}; the index now holds {len(store)} vector(s)")


In [ ]:
pip install fastapi uvicorn

In [14]:
"""Local HTTP API (FastAPI + uvicorn, in a background thread so the notebook stays usable).

    POST /similar               {"text": "...", "k": 10}  -> best-first similar situations
    GET  /situations/{key}      -> the situation's text file (404 if there isn't one)
    GET  /docs                  -> interactive docs

Uses the notebook's find_similar, text_path, TOP_K. Re-running this cell restarts the server.
"""
import os
import threading
import time
import traceback

import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field

API_HOST = "127.0.0.1"        # localhost only
API_PORT = 8000

app = FastAPI(title="Situation search")
app.add_middleware(           # lets a page served from another localhost port call this API
    CORSMiddleware,
    allow_origin_regex=r"https?://(localhost|127\.0\.0\.1)(:\d+)?",
    allow_methods=["*"],
    allow_headers=["*"],
)


class SimilarRequest(BaseModel):
    text: str = Field(min_length=1)
    k: int = Field(TOP_K, ge=1, le=100)


# Plain `def` endpoints: FastAPI runs them in a worker thread, so the blocking
# Gemini calls inside find_similar don't stall the server.
@app.post("/similar")
def similar(req: SimilarRequest):
    try:
        found = find_similar(req.text, k=req.k)
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(status_code=502, detail=f"{type(e).__name__}: {e}")
    return {"results": [
        {
            "key": int(r["key"]),                 # numpy uint64 -> int for JSON
            "distance": r["distance"],
            "relevance": r["relevance"],
            "timestamp": r["timestamp"],
            "recency": r["recency"],
            "score": r["score"],
            "file": f"/situations/{int(r['key'])}",
        }
        for r in found
    ]}


@app.get("/situations/{key}")
def situation_file(key: int):
    path = text_path(key)                         # int key, so no path traversal
    if not os.path.exists(path):
        raise HTTPException(status_code=404, detail=f"No text file for situation {key}")
    return FileResponse(path, media_type="text/plain; charset=utf-8", filename=f"{key}.txt")


# ------------------------------------------------------------ start / stop

_api = globals().get("_api") or {"server": None, "thread": None}


def stop_api():
    server, thread = _api["server"], _api["thread"]
    if server is not None:
        server.should_exit = True
    if thread is not None:
        thread.join(timeout=10)
    _api["server"] = _api["thread"] = None


def start_api(host=API_HOST, port=API_PORT):
    stop_api()                                    # also handles a server from an earlier run
    server = uvicorn.Server(uvicorn.Config(app, host=host, port=port,
                                           log_level="warning", loop="asyncio"))
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):                          # wait up to ~10 s for startup
        if server.started:
            break
        if not thread.is_alive():
            raise RuntimeError(f"API failed to start (is port {port} already in use?)")
        time.sleep(0.1)
    _api["server"], _api["thread"] = server, thread
    print(f"API running at http://{host}:{port}  (docs: http://{host}:{port}/docs)")


start_api()
# stop_api()   # uncomment to shut it down


API running at http://127.0.0.1:8000  (docs: http://127.0.0.1:8000/docs)
